# 07. Customer Feature Engineering

## Objective

The objective of this notebook is to create customer-level features that will be used for customer churn prediction and dashboard analytics.

This notebook performs:
- Loading the customer segmentation dataset
- Creating customer behavior features
- Preparing customer-level analytics
- Saving the engineered customer features

### Input
- customer_segments.csv

### Output
- customer_features.csv

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

## 2. Load Customer Segmentation Dataset

In [2]:
# Load customer segmentation dataset
df = pd.read_csv("../data/processed/customer_segments.csv")

print("Dataset loaded successfully!")

print("\nDataset Shape:")
print(df.shape)

display(df.head())

Dataset loaded successfully!

Dataset Shape:
(5878, 6)


,CustomerID,Recency,Frequency,Monetary,Cluster,CustomerSegment
0,12346.0,326,12,77556.46,0,Regular Customers
1,12347.0,2,8,4921.53,0,Regular Customers
2,12348.0,75,5,2019.40,0,Regular Customers
3,12349.0,19,4,4428.69,0,Regular Customers
4,12350.0,310,1,334.40,1,At-Risk Customers


# Load customer segmentation dataset
df = pd.read_csv("../data/processed/customer_segments.csv")

print("Dataset loaded successfully!")

print("\nDataset Shape:")
print(df.shape)

display(df.head())

In [3]:
print("=" * 60)
print("Dataset Information")
print("=" * 60)

df.info()

print("\nDataset Shape:", df.shape)

print("\nMissing Values:", df.isnull().sum().sum())

print("Duplicate Rows:", df.duplicated().sum())

Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 5878 entries, 0 to 5877
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       5878 non-null   float64
 1   Recency          5878 non-null   int64  
 2   Frequency        5878 non-null   int64  
 3   Monetary         5878 non-null   float64
 4   Cluster          5878 non-null   int64  
 5   CustomerSegment  5878 non-null   str    
dtypes: float64(2), int64(3), str(1)
memory usage: 275.7 KB

Dataset Shape: (5878, 6)

Missing Values: 0
Duplicate Rows: 0


In [4]:
print(df.columns.tolist())

['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Cluster', 'CustomerSegment']


In [7]:
transactions = pd.read_csv(
    "../data/processed/online_retail_II_feature_engineered.csv"
)

print(transactions.shape)
transactions.head()

(779425, 20)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Year,Month,MonthName,Quarter,Day,DayOfWeek,Hour,Revenue,TimeOfDay,Season,BasketSize,UniqueProducts
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009,12,December,4,1,Tuesday,7,83.4,Morning,Winter,166,8
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009,12,December,4,1,Tuesday,7,81.0,Morning,Winter,166,8
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009,12,December,4,1,Tuesday,7,81.0,Morning,Winter,166,8
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009,12,December,4,1,Tuesday,7,100.8,Morning,Winter,166,8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009,12,December,4,1,Tuesday,7,30.0,Morning,Winter,166,8


In [8]:
segments = df.copy()

print(segments.shape)

(5878, 6)


In [9]:
print(transactions.shape)

print(segments.shape)

(779425, 20)
(5878, 6)


## 5. Customer Feature Engineering

In [11]:
# Convert InvoiceDate to datetime
transactions["InvoiceDate"] = pd.to_datetime(transactions["InvoiceDate"])

print("InvoiceDate converted successfully.")

InvoiceDate converted successfully.


In [12]:
customer_features = (
    transactions
    .groupby("Customer ID")
    .agg(
        FirstPurchaseDate=("InvoiceDate", "min"),
        LastPurchaseDate=("InvoiceDate", "max"),
        TotalInvoices=("Invoice", "nunique"),
        TotalQuantity=("Quantity", "sum"),
        TotalRevenue=("Revenue", "sum"),
        AvgOrderValue=("Revenue", "mean"),
        AvgBasketSize=("BasketSize", "mean"),
        AvgUniqueProducts=("UniqueProducts", "mean"),
        PreferredPurchaseHour=("Hour", lambda x: x.mode()[0]),
        PreferredSeason=("Season", lambda x: x.mode()[0]),
        Country=("Country", lambda x: x.mode()[0])
    )
    .reset_index()
)

print("Customer-level features created successfully!")

Customer-level features created successfully!


In [13]:
print("Shape:", customer_features.shape)

customer_features.head()

Shape: (5878, 12)


,Customer ID,FirstPurchaseDate,LastPurchaseDate,TotalInvoices,TotalQuantity,TotalRevenue,AvgOrderValue,AvgBasketSize,AvgUniqueProducts,PreferredPurchaseHour,PreferredSeason,Country
0,12346.0,2009-12-14 08:34:00,2011-01-18 10:01:00,12,74285,77556.46,2281.072353,2195.500000,11.647059,13,Summer,United Kingdom
1,12347.0,2010-10-31 14:20:00,2011-12-07 15:52:00,8,2967,4921.53,22.169054,425.594595,32.054054,14,Autumn,Iceland
2,12348.0,2010-09-27 14:59:00,2011-09-25 13:13:00,5,2714,2019.40,39.596078,674.117647,13.549020,14,Autumn,Finland
3,12349.0,2010-04-29 13:20:00,2011-11-21 09:51:00,4,1624,4428.69,25.306800,550.668571,59.834286,9,Autumn,Italy
4,12350.0,2011-02-02 16:01:00,2011-02-02 16:01:00,1,197,334.40,19.670588,197.000000,17.000000,16,Winter,Norway


## 6. Merge Customer Features

In [14]:
# Rename column for merging
customer_features.rename(
    columns={"Customer ID": "CustomerID"},
    inplace=True
)

# Merge customer features with customer segments
customer_features = customer_features.merge(
    segments,
    on="CustomerID",
    how="left"
)

print("Customer features merged successfully!")

print("Shape:", customer_features.shape)

customer_features.head()

Customer features merged successfully!
Shape: (5878, 17)


,CustomerID,FirstPurchaseDate,LastPurchaseDate,TotalInvoices,TotalQuantity,TotalRevenue,AvgOrderValue,AvgBasketSize,AvgUniqueProducts,PreferredPurchaseHour,PreferredSeason,Country,Recency,Frequency,Monetary,Cluster,CustomerSegment
0,12346.0,2009-12-14 08:34:00,2011-01-18 10:01:00,12,74285,77556.46,2281.072353,2195.500000,11.647059,13,Summer,United Kingdom,326,12,77556.46,0,Regular Customers
1,12347.0,2010-10-31 14:20:00,2011-12-07 15:52:00,8,2967,4921.53,22.169054,425.594595,32.054054,14,Autumn,Iceland,2,8,4921.53,0,Regular Customers
2,12348.0,2010-09-27 14:59:00,2011-09-25 13:13:00,5,2714,2019.40,39.596078,674.117647,13.549020,14,Autumn,Finland,75,5,2019.40,0,Regular Customers
3,12349.0,2010-04-29 13:20:00,2011-11-21 09:51:00,4,1624,4428.69,25.306800,550.668571,59.834286,9,Autumn,Italy,19,4,4428.69,0,Regular Customers
4,12350.0,2011-02-02 16:01:00,2011-02-02 16:01:00,1,197,334.40,19.670588,197.000000,17.000000,16,Winter,Norway,310,1,334.40,1,At-Risk Customers


In [15]:
print(customer_features.columns.tolist())

['CustomerID', 'FirstPurchaseDate', 'LastPurchaseDate', 'TotalInvoices', 'TotalQuantity', 'TotalRevenue', 'AvgOrderValue', 'AvgBasketSize', 'AvgUniqueProducts', 'PreferredPurchaseHour', 'PreferredSeason', 'Country', 'Recency', 'Frequency', 'Monetary', 'Cluster', 'CustomerSegment']


## 7. Save Customer Features Dataset

In [16]:
customer_features.to_csv(
    "../data/processed/customer_features.csv",
    index=False
)

print("customer_features.csv saved successfully!")

customer_features.csv saved successfully!


## 8. Business Insights

### Key Findings

- Created a customer-level feature dataset by combining transaction history with customer segmentation.
- Aggregated customer purchasing behavior, including total revenue, order frequency, and preferred purchasing patterns.
- Combined behavioral features with RFM metrics and customer segments.
- The resulting dataset will serve as the input for Customer Churn Prediction in Notebook 08.

## 9. Summary

This notebook successfully transformed transaction-level retail data into a customer-level feature dataset.

Output Generated:
- customer_features.csv

Next Notebook:
- 08_Customer_Churn_Prediction.ipynb

In [6]:
print(df.shape)
print(df.columns.tolist())

(5878, 6)
['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Cluster', 'CustomerSegment']
